# 🏆 Amazon ML Challenge 2026: Master Google Colab Pipeline
**Task:** Cross-Source Business Entity Resolution (`S2-` and `S3-` $\to$ `S1-`)  
**Metric:** Macro $F_{0.5}$ Score (Precision-Weighted)  
**Hardware:** NVIDIA T4 GPU / High-RAM CPU Accelerated  

---
### ⚡ 3-Step Colab Execution:
1. **Set Runtime:** Click **Runtime $\to$ Change runtime type $\to$ T4 GPU** (or High-RAM CPU).
2. **Fill in URLs:** Enter your GitHub repo URL in **Step 2** and Dataset direct download link in **Step 4**.
3. **Run All:** Click **Runtime $\to$ Run all**. The notebook will train the ensemble, run full 1.73M test inference in ~4-6 minutes, validate submission files, and auto-download `submission_package.zip`.

## Step 1: System Hardware & GPU Verification

In [ ]:
import torch
import os

print("=== Hardware Specs ===")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "CUDA not active (Running CPU mode)"
!free -h

has_gpu = torch.cuda.is_available()
print(f"\nPyTorch CUDA Available: {has_gpu}")
if has_gpu:
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Clean Pipeline Repository

In [ ]:
# REPLACE WITH YOUR GITHUB REPOSITORY URL
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}
!pwd

## Step 3: Install Fast Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("✓ All required libraries installed successfully!")

## Step 4: Download & Extract Dataset

In [ ]:
# REPLACE WITH YOUR DIRECT DATASET ZIP DOWNLOAD LINK (e.g. S3, Google Drive, or Server URL)
DATASET_ZIP_URL = "https://YOUR_DATASET_DIRECT_DOWNLOAD_LINK_HERE"

dataset_dir = "dataset/student_resource/dataset"
if not os.path.exists(os.path.join(dataset_dir, "test", "test_source1.tsv")):
    os.makedirs("dataset/student_resource", exist_ok=True)
    if DATASET_ZIP_URL.startswith("http"):
        print("Downloading dataset package...")
        !wget -q --show-progress "{DATASET_ZIP_URL}" -O dataset.zip
        print("Extracting dataset...")
        !unzip -q dataset.zip -d dataset/student_resource/
        print("✓ Dataset ready!")
    else:
        print("⚠️ Please provide your valid dataset download link in DATASET_ZIP_URL")
else:
    print("✓ Dataset files detected in workspace.")

## Step 5: Multi-Model Training & Threshold Calibration (80/20 Holdout Split)
Trains LightGBM, XGBoost, CatBoost, and Logistic Regression on cohesive training pairs and tunes $\tau^*$ on 20% holdout split.

In [ ]:
import sys
sys.path.insert(0, os.path.abspath("."))

import src
from src.data.validation_split import ValidationSplitManager

# Ensure isolated split exists
split_mgr = ValidationSplitManager()
split_mgr.create_isolated_split(train_ratio=0.80, random_state=42)

# Train models with GPU acceleration (if CUDA active)
artifacts = src.train_and_validate_pipeline(
    train_s1_samples=10000,
    val_s1_samples=2500,
    random_state=42,
    save_path="models/pipeline_artifacts.joblib",
    load_cached=True
)

print("\n=== Benchmark Performance on 20% Holdout Split ===")
display(artifacts["benchmark_summary"])

## Step 6: Full Streaming Test Set Inference (1,732,544 Entities)
Runs high-throughput streaming candidate generation (Inverted Index), 16 RapidFuzz tabular features, model predictions, and global injective matching across France, US, and India.

In [ ]:
# Runs full test streaming inference
!python3 generate_final_submission.py

## Step 7: Official Submission Validation
Verifies submission formatting, column schemas, entity row counts, and candidate subset rules using the official challenge validator.

In [ ]:
!python3 dataset/student_resource/utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/student_resource/dataset/test

## Step 8: Package and Auto-Download Submission Package

In [ ]:
import zipfile
from google.colab import files

submission_zip = "submission_package.zip"

with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Scored output TSV files
    if os.path.exists("output/matching_results.tsv"):
        zipf.write("output/matching_results.tsv", arcname="output/matching_results.tsv")
    if os.path.exists("output/candidate_pairs.tsv"):
        zipf.write("output/candidate_pairs.tsv", arcname="output/candidate_pairs.tsv")
    
    # 2. Documentation
    if os.path.exists("DOCUMENTATION.md"):
        zipf.write("DOCUMENTATION.md", arcname="DOCUMENTATION.md")
    
    # 3. Source code for reproducibility
    for root, dirs, filenames in os.walk("src"):
        for filename in filenames:
            if filename.endswith((".py", ".md")):
                filepath = os.path.join(root, filename)
                zipf.write(filepath, arcname=filepath)

zip_size_mb = os.path.getsize(submission_zip) / (1024 * 1024)
print(f"✓ Submission package created: {submission_zip} ({zip_size_mb:.2f} MB)")

# Trigger browser download
files.download(submission_zip)